# Stroke Risk — Exploratory Data Analysis

**Goal:** understand who is at risk of stroke and find the data issues we must handle before modelling.

> **Leak-free rule:** this notebook looks at the **training split only**. The validation and test sets stay untouched so our later evaluation stays honest.

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from stroke_risk import data
from stroke_risk import plotting as pl

pl.set_theme()

# Explore the TRAINING split only.
splits = data.split_data(data.load_raw())
train = splits.train
train.shape

## 1. The dataset at a glance

In [ ]:
train.info()
train.head()

## 2. Stroke is rare

This class imbalance is the central challenge of the whole project.

In [ ]:
rate = train['stroke'].mean()

fig, ax = plt.subplots(figsize=(9, 1.9))
ax.barh(0, (1 - rate) * 100, color=pl.MUTED)
ax.barh(0, rate * 100, left=(1 - rate) * 100, color=pl.ACCENT)

ax.text((1 - rate) * 50, 0, f'No stroke   {100 * (1 - rate):.1f}%',
        ha='center', va='center', color='white', fontsize=11, fontweight='bold')
ax.text(100, 0.75, f'Stroke   {100 * rate:.1f}%',
        ha='right', va='bottom', color=pl.ACCENT, fontsize=11, fontweight='bold')

ax.set_xlim(0, 100)
ax.set_ylim(-0.6, 0.6)
ax.axis('off')
pl.add_titles(ax, 'Only about 1 in 20 patients had a stroke',
              'Share of patients by outcome (training set)')
plt.show()

## 3. Missing data

In [ ]:
miss = train.isna().mean().mul(100)
miss = miss[miss > 0].sort_values()

fig, ax = plt.subplots(figsize=(7, 2.2))
bars = ax.barh(miss.index, miss.values, color=pl.ACCENT)
ax.bar_label(bars, fmt='%.1f%%', padding=4, color=pl.SUBTLE, fontsize=10)
ax.set_xticks([])
pl.despine(ax, left=False, bottom=True)
pl.add_titles(ax, 'Only BMI has missing values',
              'Share of missing entries per column')
plt.show()

## 4. Age is the dominant risk factor

In [ ]:
bins = [0, 20, 30, 40, 50, 60, 70, 80, 120]
labels = ['<20', '20s', '30s', '40s', '50s', '60s', '70s', '80+']
band = pd.cut(train['age'], bins=bins, labels=labels, right=False)
rate_by_age = train.groupby(band, observed=True)['stroke'].mean().mul(100)

fig, ax = plt.subplots()
bars = ax.bar(rate_by_age.index.astype(str), rate_by_age.values, color=pl.MUTED)
for b, label in zip(bars, rate_by_age.index):
    if label in ('70s', '80+'):
        b.set_color(pl.ACCENT)
ax.bar_label(bars, fmt='%.0f%%', padding=3, color=pl.SUBTLE, fontsize=9)
ax.set_yticks([])
pl.despine(ax, left=True)
pl.add_titles(ax, 'Stroke risk climbs steeply with age',
              'Percentage of patients in each age group who had a stroke')
plt.show()

## 5. Glucose and BMI

In [ ]:
features = [('age', 'Age'), ('avg_glucose_level', 'Average glucose level'), ('bmi', 'BMI')]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, (col, name) in zip(axes, features):
    for cls in (0, 1):
        vals = train.loc[train['stroke'] == cls, col].dropna()
        ax.hist(vals, bins=30, density=True, color=pl.STROKE_COLORS[cls],
                alpha=0.85 if cls == 1 else 0.6, label=pl.STROKE_LABELS[cls])
    ax.set_title(name, loc='left', fontsize=12, fontweight='bold', color=pl.INK)
    ax.set_yticks([])
    pl.despine(ax, left=True)
axes[0].legend(loc='upper right')
fig.text(0, 1.02, 'Stroke patients skew toward higher age, glucose and BMI',
         fontsize=15, fontweight='bold', color=pl.INK)
plt.tight_layout()
plt.show()

## 6. Comorbidities

In [ ]:
conditions = [('hypertension', 'Hypertension'), ('heart_disease', 'Heart disease')]
names, values, colors = [], [], []
for col, name in conditions:
    for present in (0, 1):
        names.append(f"{name}\n{'Yes' if present else 'No'}")
        values.append(train.loc[train[col] == present, 'stroke'].mean() * 100)
        colors.append(pl.ACCENT if present else pl.MUTED)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names, values, color=colors)
ax.bar_label(bars, fmt='%.1f%%', padding=3, color=pl.SUBTLE, fontsize=10)
ax.set_yticks([])
pl.despine(ax, left=True)
pl.add_titles(ax, 'Existing conditions multiply stroke risk',
              'Stroke rate with and without each condition')
plt.show()

## 7. Data quirks to handle

In [ ]:
print('gender          :', dict(train['gender'].value_counts()))
print('work_type       :', dict(train['work_type'].value_counts()))
print('smoking_status  :', dict(train['smoking_status'].value_counts()))
print('age  min / max  :', train['age'].min(), '/', train['age'].max())
print('bmi missing     :', int(train['bmi'].isna().sum()))

## Findings → pipeline decisions

_To be filled in from the charts above once we run them:_

- **Target imbalance (~5%)** → use stratified splits (done) and handle imbalance during modelling.
- **`bmi` missing** → impute with **train-set median** (fit on train only).
- **`id`** → drop (identifier, no predictive value).
- **Quirks to confirm:** rare `gender` category, `age` stored as fractions, `smoking_status = "Unknown"`.
- **Strongest signals so far:** age, glucose, BMI, hypertension, heart disease.